# 🧠 The Perceptron: The Foundation of Neural Networks

Welcome to the hands-on explanation notebook for the **Perceptron**! In this notebook, we will:
1. Formulate the Perceptron model and Heaviside Step Activation.
2. Implement a complete Perceptron classifier from scratch.
3. Train the Perceptron to learn logical gates: `AND`, `OR`, and `XOR`.
4. Plot the training points and the **linear decision boundary** to visualize successful separation.
5. Demonstrate why a Perceptron fails to solve the `XOR` gate due to the **Linear Separability Limit**.
6. Discuss the transition from single neurons to Multi-Layer Perceptrons (MLPs) and modern deep networks (such as YOLO's fully connected and convolutional layers).

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. Implementing the Perceptron Class from Scratch

Let's build a Perceptron that tracks weights, bias, and learning rate, updating them on errors.

In [ ]:
class ScratchPerceptron:
    def __init__(self, input_dim=2, lr=0.1):
        self.weights = np.zeros(input_dim)
        self.bias = 0.0
        self.lr = lr
        
    def predict(self, x):
        z = np.dot(x, self.weights) + self.bias
        return 1 if z >= 0 else 0
        
    def train(self, X, y, epochs=15):
        history = []
        for epoch in range(epochs):
            errors = 0
            for xi, yi in zip(X, y):
                y_pred = self.predict(xi)
                error = yi - y_pred
                if error != 0:
                    self.weights += self.lr * error * xi
                    self.bias += self.lr * error
                    errors += 1
            history.append(errors)
            if errors == 0:
                break
        return history

## 2. Solving AND and OR Logical Gates

Let's train two instances of our perceptron on `AND` and `OR` data, then print the learned parameters.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

y_and = np.array([0, 0, 0, 1])
y_or = np.array([0, 1, 1, 1])

p_and = ScratchPerceptron()
hist_and = p_and.train(X, y_and, epochs=20)

p_or = ScratchPerceptron()
hist_or = p_or.train(X, y_or, epochs=20)

print("AND Weights:", p_and.weights, "| Bias:", p_and.bias)
print("OR Weights :", p_or.weights, "| Bias:", p_or.bias)

## 3. Visualizing Decision Boundaries (AND vs. OR vs. XOR)

Let's write a plotting function that draws the decision boundary line:
$$w_1 x_1 + w_2 x_2 + b = 0 \implies x_2 = -\frac{w_1}{w_2} x_1 - \frac{b}{w_2}$$

For the `XOR` gate, we will show that it is impossible to separate the classes.

In [ ]:
y_xor = np.array([0, 1, 1, 0])
p_xor = ScratchPerceptron()
p_xor.train(X, y_xor, epochs=50)

def plot_gate_boundary(ax, p, X, y, title):
    for xi, yi in zip(X, y):
        marker = 'o' if yi == 1 else 'x'
        color = 'green' if yi == 1 else 'red'
        ax.scatter(xi[0], xi[1], color=color, marker=marker, s=150, linewidth=3, zorder=5)
        
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('Input 1 (x1)')
    ax.set_ylabel('Input 2 (x2)')
    ax.set_title(title)
    ax.grid(True, linestyle='--', alpha=0.5)
    
    w1, w2 = p.weights[0], p.weights[1]
    b = p.bias
    
    if w2 != 0:
        x1_vals = np.linspace(-0.5, 1.5, 100)
        x2_vals = -(w1 * x1_vals + b) / w2
        ax.plot(x1_vals, x2_vals, color='blue', linewidth=2.5, label='Decision Boundary')
        ax.legend()
    else:
        ax.text(0.5, 0.5, 'Boundary Undefined', color='purple', ha='center', fontsize=12, bbox=dict(facecolor='white', alpha=0.8))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_gate_boundary(axes[0], p_and, X, y_and, "AND Gate (Linearly Separable)")
plot_gate_boundary(axes[1], p_or, X, y_or, "OR Gate (Linearly Separable)")
plot_gate_boundary(axes[2], p_xor, X, y_xor, "XOR Gate (Failed - Non-Separable)")
plt.tight_layout()
plt.show()

Look at the plots:
-   **AND and OR gates:** The blue line divides the inputs perfectly.
-   **XOR gate:** The green circles (1s) are at $(0,1)$ and $(1,0)$, while the red crosses (0s) are at $(0,0)$ and $(1,1)$. There is no possible straight line that can place green circles on one side and red crosses on the other!

## 💡 Connection to YOLO and Modern Deep Learning
*   **The Multi-Layer Solution:** To solve XOR, we need to stack perceptrons into multiple layers (Multi-Layer Perceptron) with non-linear activation functions (like Sigmoid or ReLU).
*   **Modern Layers:** In YOLO models, classification and detection heads use layers of interconnected neurons (Linear layers followed by SiLU activation) to draw highly complex, curved decision boundaries in multi-dimensional space, enabling the network to classify overlapping objects.